# 9-Persona DiffusionDrive Inference Visualization

Checkpoint: `ckpt/9persona/9persona_diffusiondrive_hier.ckpt`

9 persona categories:
- **UH**: Urgency High, **UM**: Urgency Medium, **UL**: Urgency Low
- **CL**: Caution Low, **CM**: Caution Medium, **CH**: Caution High

## 1. Scene Loader

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import hydra
from hydra.utils import instantiate

from navsim.common.dataloader import SceneLoader
from navsim.common.dataclasses import SceneFilter, SensorConfig

SPLIT = "test"
FILTER = "navtest"
OPENSCENE_DATA_ROOT = "/data/leechan/2025/datasets/openscene"
PER_SCENE_DIR = "/data/leechan/2025/gpt_generation/processed_results/per_scene"

hydra.initialize(config_path="../navsim/planning/script/config/common/train_test_split/scene_filter")
cfg = hydra.compose(config_name=FILTER)
scene_filter: SceneFilter = instantiate(cfg)
openscene_data_root = Path(OPENSCENE_DATA_ROOT)

scene_loader = SceneLoader(
    openscene_data_root / f"navsim_logs/{SPLIT}",
    openscene_data_root / f"sensor_blobs/{SPLIT}",
    scene_filter,
    sensor_config=SensorConfig.build_all_sensors(),
)
print(f"Loaded {len(scene_loader.tokens)} scenes")

/tmp/ipykernel_3138398/3829508691.py:20: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path="../navsim/planning/script/config/common/train_test_split/scene_filter")
Loading logs: 100%|██████████| 136/136 [00:04<00:00, 27.89it/s]

Loaded 12146 scenes


## 2. Load Model

In [2]:
import torch, hydra
from pathlib import Path
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from transformers import AutoTokenizer

REPO_ROOT = Path("..").resolve()
CFG_ROOT = "../navsim/planning/script/config"
sys.path.extend([str(REPO_ROOT), str(REPO_ROOT / "navsim")])

if os.getcwd().endswith("vis_ipynb"):
    os.chdir("..")

CKPT = Path("ckpt/9persona/9persona_diffusiondrive_hier.ckpt")

if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
hydra.initialize(config_path=str(f"{CFG_ROOT}/pdm_scoring"))

overrides = [
    "agent=diffusiondrive_agent",
    f"agent.checkpoint_path={CKPT}",
    "agent.lr=0.0",
    "train_test_split=navtest",
    "worker=ray_distributed",
]
cfg = hydra.compose(config_name="default_run_pdm_score", overrides=overrides)

agent = instantiate(cfg.agent)
agent.eval()
print("Model loaded")

/tmp/ipykernel_3138398/2878854667.py:18: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path=str(f"{CFG_ROOT}/pdm_scoring"))


Missing keys when loading pretrained weights: ['_transfuser_model._trajectory_head.alpha_proj.weight', '_transfuser_model._trajectory_head.alpha_proj.bias']
Unexpected keys when loading pretrained weights: ['_transfuser_model._trajectory_head.urgency_proj.0.weight', '_transfuser_model._trajectory_head.urgency_proj.0.bias', '_transfuser_model._trajectory_head.urgency_proj.2.weight', '_transfuser_model._trajectory_head.urgency_proj.2.bias', '_transfuser_model._trajectory_head.comfort_proj.0.weight', '_transfuser_model._trajectory_head.comfort_proj.0.bias', '_transfuser_model._trajectory_head.comfort_proj.2.weight', '_transfuser_model._trajectory_head.comfort_proj.2.bias', '_transfuser_model._trajectory_head.alpha_urgency.weight', '_transfuser_model._trajectory_head.alpha_urgency.bias', '_transfuser_model._trajectory_head.alpha_comfort.0.weight', '_transfuser_model._trajectory_head.alpha_comfort.0.bias', '_transfuser_model._trajectory_head.alpha_comfort.2.weight', '_transfuser_model._traj

In [ ]:
# Tokenizer
bert_backbone = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(bert_backbone)

def tokenizing(text):
    tok = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=100,
        return_tensors="pt",
    )
    return {
        "bert_feature": [tok["input_ids"].tolist()[0]],
        "attention_mask": [tok["attention_mask"].tolist()[0]],
    }

# JSON key (per_scene JSON에서 사용하는 키)
CAT_KEYS = [
    "UH_CL", "UH_CM", "UH_CH",
    "UM_CL", "UM_CM", "UM_CH",
    "UL_CL", "UL_CM", "UL_CH",
]

# Display name (시각화용)
CAT_NAMES = [
    "Urg. High - Comf. Low", "Urg. High - Comf. Mid", "Urg. High - Comf. High",
    "Urg. Mid - Comf. Low",  "Urg. Mid - Comf. Mid",  "Urg. Mid - Comf. High",
    "Urg. Low - Comf. Low",  "Urg. Low - Comf. Mid",  "Urg. Low - Comf. High",
]

# key -> display name 매핑
KEY_TO_NAME = dict(zip(CAT_KEYS, CAT_NAMES))

# UH: red gradient, UM: blue gradient, UL: green gradient
# CL -> CM -> CH: dark -> light
CAT_COLORS = {
    "UH_CL": "#E85050", "UH_CM": "#ED7373", "UH_CH": "#F19696",
    "UM_CL": "#5090D0", "UM_CM": "#73A6D9", "UM_CH": "#96BCE3",
    "UL_CL": "#50B870", "UL_CM": "#73C68D", "UL_CH": "#96D4A9",
}

## 3. Inference Helpers

In [3]:
def get_per_scene_data(token):
    path = os.path.join(PER_SCENE_DIR, f"{token}.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def run_inference_category(scene, category, scene_data):
    """Run inference for a single category using scene.get_agent_input_uc()."""
    cat_data = scene_data[category]
    pre_tok = cat_data.get("tokenized", {}).get(bert_backbone)
    if pre_tok is not None:
        uc = {
            "bert_feature": [pre_tok["input_ids"]],
            "attention_mask": [pre_tok["attention_mask"]],
        }
    else:
        uc = tokenizing(cat_data["user_intent"])
    trajectory = agent.compute_trajectory(scene.get_agent_input_uc(uc))
    return trajectory, cat_data.get("user_intent", "")


def run_inference_free_text(scene, text):
    """Run inference with arbitrary free-form text."""
    uc = tokenizing(text)
    return agent.compute_trajectory(scene.get_agent_input_uc(uc))

## 4. Visualization Imports

In [4]:
from navsim.visualization.plots import (
    plot_bev_with_agent_pred_uc_,
    plot_bev_with_agent_human,
    plot_bev_with_agent_uc_comb2_,
    plot_stitched_cameras_frame_,
    configure_bev_ax,
    configure_ax,
)
from navsim.visualization.config import BEV_PLOT_CONFIG, TRAJECTORY_CONFIG
from navsim.visualization.bev import add_configured_bev_on_ax, add_trajectory_to_bev_ax
import cv2

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 5. Pick Scene & Visualize All 9 Personas
이 셀을 반복 실행하면 매번 새로운 씬을 뽑아서 시각화합니다.
마음에 드는 결과가 나올 때까지 반복 실행하세요.

## 5. Batch Filtering & Save
조건 (완화):
1. **같은 Urgency 내**: CL > CM > CH (3그룹)
2. **같은 Caution에서**: UH > UM > UL (3그룹)
3. 어떤 trajectory든 후진(x < 0인 timestep)이 있으면 제외

통과한 씬만 `tmp/` 폴더에 3x3 그리드로 저장합니다.

In [ ]:
os.makedirs("tmp", exist_ok=True)

def total_distance(traj):
    """trajectory의 총 이동거리 (유클리드 누적)."""
    poses = traj.poses[:, :2]  # (T, 2) - x, y
    diffs = np.diff(poses, axis=0)
    return float(np.sum(np.linalg.norm(diffs, axis=1)))

def has_backward(traj):
    """trajectory에 후진이 있는지 (x < 0인 timestep이 있으면 True)."""
    return bool(np.any(traj.poses[:, 0] < 0))

def check_distance_order(pred_trajectories):
    """
    완화된 순서 체크:
    1) 같은 Urgency 내: CL > CM > CH  (3그룹)
    2) 같은 Caution에서: UH > UM > UL (3그룹)
    """
    dists = {k: total_distance(pred_trajectories[k]) for k in CAT_KEYS}

    # 같은 Urgency 내: CL > CM > CH
    for u in ["UH", "UM", "UL"]:
        if not (dists[f"{u}_CL"] > dists[f"{u}_CM"] > dists[f"{u}_CH"]):
            return False, dists

    # 같은 Caution에서: UH > UM > UL
    for c in ["CL", "CM", "CH"]:
        if not (dists[f"UH_{c}"] > dists[f"UM_{c}"] > dists[f"UL_{c}"]):
            return False, dists

    return True, dists

saved = 0
skipped_no_json = 0
skipped_backward = 0
skipped_order = 0
skipped_fail = 0

for idx, token in enumerate(scene_loader.tokens):
    if idx % 500 == 0:
        print(f"Processing {idx}/{len(scene_loader.tokens)} | saved={saved}")

    # per_scene JSON 존재 확인
    json_path = os.path.join(PER_SCENE_DIR, f"{token}.json")
    if not os.path.exists(json_path):
        skipped_no_json += 1
        continue

    scene = scene_loader.get_scene_from_token(token)
    frame_idx = scene.scene_metadata.num_history_frames - 1
    scene_data = get_per_scene_data(token)

    # 9개 전부 inference
    pred_trajectories = {}
    ok = True
    for key in CAT_KEYS:
        try:
            traj, _ = run_inference_category(scene, key, scene_data)
            pred_trajectories[key] = traj
        except:
            ok = False
            break
    if not ok or len(pred_trajectories) < 9:
        skipped_fail += 1
        continue

    # 후진 체크: 9개 중 하나라도 후진이 있으면 제외
    if any(has_backward(pred_trajectories[k]) for k in CAT_KEYS):
        skipped_backward += 1
        continue

    # 이동거리 순서 체크 (완화)
    order_ok, dists = check_distance_order(pred_trajectories)
    if not order_ok:
        skipped_order += 1
        continue

    # 통과! 3x3 grid 저장
    human_trajectory = scene.get_future_trajectory()
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    for i, key in enumerate(CAT_KEYS):
        row, col = i // 3, i % 3
        ax = axes[row, col]
        add_configured_bev_on_ax(ax, scene.map_api, scene.frames[frame_idx])
        add_trajectory_to_bev_ax(ax, human_trajectory, TRAJECTORY_CONFIG["human"])
        color = CAT_COLORS[key]
        traj_config = {
            "fill_color": color, "fill_color_alpha": 1.0,
            "line_color": color, "line_color_alpha": 1.0,
            "line_width": 2.5, "line_style": "-",
            "marker": "o", "marker_size": 5,
            "marker_edge_color": "black", "zorder": 3,
        }
        add_trajectory_to_bev_ax(ax, pred_trajectories[key], traj_config)
        configure_bev_ax(ax); configure_ax(ax)
        ax.set_title(f"{KEY_TO_NAME[key]} (d={dists[key]:.1f}m)", fontsize=10, fontweight="bold")
    plt.tight_layout()
    fig.savefig(f"tmp/{saved:04d}_{token}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved += 1

print(f"\n===== Done =====")
print(f"Total scenes: {len(scene_loader.tokens)}")
print(f"Saved: {saved}")
print(f"Skipped (no JSON): {skipped_no_json}")
print(f"Skipped (inference fail): {skipped_fail}")
print(f"Skipped (backward): {skipped_backward}")
print(f"Skipped (order): {skipped_order}")

## 9. Free-Text Inference

In [ ]:
custom_text = "Please drive carefully, there is a school zone ahead."

custom_traj = run_inference_free_text(scene, custom_text)
fig, ax = plot_bev_with_agent_pred_uc_(scene, custom_traj, intent="agent")
ax.set_title(f"Free text: \"{custom_text[:60]}\"", fontsize=9)
plt.tight_layout()
plt.show()

## 10. Multi-Scene Loop

In [ ]:
N_SCENES = 5
REPR_CATS = ["UH_CM", "UM_CM", "UL_CM"]
REPR_STYLES = ["emergency", "normal", "relaxed"]

random_tokens = np.random.choice(scene_loader.tokens, size=N_SCENES, replace=False)

for t_idx, tok in enumerate(random_tokens):
    json_path = os.path.join(PER_SCENE_DIR, f"{tok}.json")
    if not os.path.exists(json_path):
        print(f"[{t_idx+1}/{N_SCENES}] Skipping {tok} (no per_scene JSON)")
        continue

    sc = scene_loader.get_scene_from_token(tok)
    fi = sc.scene_metadata.num_history_frames - 1
    ht = sc.get_future_trajectory()
    sd = get_per_scene_data(tok)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Camera view
    frame = sc.frames[fi]
    l0 = frame.cameras.cam_l0.image[28:-28, 416:-416]
    f0 = frame.cameras.cam_f0.image[28:-28]
    r0 = frame.cameras.cam_r0.image[28:-28, 416:-416]
    stitched = np.concatenate([l0, f0, r0], axis=1)
    resized = cv2.resize(stitched, (1024, 256))
    axes[0].imshow(resized)
    configure_ax(axes[0])
    axes[0].set_title("Camera", fontsize=9)

    for j, (cat, style) in enumerate(zip(REPR_CATS, REPR_STYLES)):
        ax = axes[j + 1]
        add_configured_bev_on_ax(ax, sc.map_api, sc.frames[fi])
        add_trajectory_to_bev_ax(ax, ht, TRAJECTORY_CONFIG["human"])
        try:
            traj, _ = run_inference_category(sc, cat, sd)
            add_trajectory_to_bev_ax(ax, traj, TRAJECTORY_CONFIG[style])
        except:
            pass
        configure_bev_ax(ax)
        configure_ax(ax)
        ax.set_title(cat, fontsize=9)

    fig.suptitle(f"Scene {t_idx+1}/{N_SCENES} | {tok[:16]}...", fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close(fig)